In [1]:
import os, sys, time
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)

import numpy as np, pandas as pd
from src.data_loader import load_data
from src.evaluation import evaluate
from src.models.item_knn import ItemKNNModel

data = load_data()
R = data["interaction_matrix"]

VERİ YÜKLENDİ — DOĞRULAMA ÖZETİ
  Etkileşim matrisi şekli  : 94745 x 25610
  Beklenen                 : 94745 x 25610
  Pozitif etkileşim (nnz)  : 504,975
  Beklenen (yaklaşık)      : ~505.000
  Val  label==1 satır      : 75,799  (beklenen: 75.799)
  Test label==1 satır      : 74,021  (beklenen: 74.021)

  SOGUK URUN (iki tanim karsilastirmasi):
    Tanim 1 — train'de HIC gorunmeyen  (KULLANILAN):
      Toplam soguk urun: 84
      Test pozitifinde : 251 satir (kullanici)  |  78 tekil urun
      Beklenen satir   : ~251
    Tanim 2 — train'de label==1 olmayan (referans):
      Toplam soguk urun: 243
      Test pozitifinde : 462 satir (kullanici)  |  184 tekil urun


In [2]:
model = ItemKNNModel(k=1000, shrink=10, pop_weight=1e-3).fit(R)
print(model, f"| fit: {model.fit_seconds:.1f}s")

t = time.time()
res = evaluate(model.score, data["val_candidates"], cold_items=data["cold_items"],
               user_activity=data["user_activity"], verbose=False)
print(f"değerlendirme: {time.time()-t:.1f}s")
print(res.keys())
res

ItemKNNModel(k=1000, shrink=10, pop_weight=0.001, W_nnz=4434260) | fit: 8.8s
değerlendirme: 27.3s
dict_keys(['overall', 'cold_user', 'activity'])


{'overall': {'hit_rate': 0.6220134830274806,
  'ndcg': 0.4407005048848438,
  'precision': 0.06220134830274806,
  'recall': 0.6220134830274806,
  'coverage_pct': 93.88910581803984,
  'coverage_n': 24045,
  'n_users': 75799},
 'cold_user': {'hit_rate': 0.0,
  'ndcg': 0.0,
  'precision': 0.0,
  'recall': 0.0,
  'coverage_pct': 4.135103475204998,
  'coverage_n': 1059,
  'n_users': 114},
 'activity': {'low': {'hit_rate': 0.619455252918288,
   'ndcg': 0.4314101399010589,
   'precision': 0.0619455252918288,
   'recall': 0.619455252918288,
   'coverage_pct': 74.77157360406092,
   'coverage_n': 19149,
   'n_users': 24415},
  'medium': {'hit_rate': 0.6188336329139205,
   'ndcg': 0.4395525273329651,
   'precision': 0.061883363291392046,
   'recall': 0.6188336329139205,
   'coverage_pct': 84.42795782897305,
   'coverage_n': 21622,
   'n_users': 30042},
  'high': {'hit_rate': 0.6294161746790367,
   'ndcg': 0.45294452150431014,
   'precision': 0.06294161746790367,
   'recall': 0.6294161746790367,
  

In [3]:
from src.models.popularity import PopularityModel

pop_model = PopularityModel().fit(R)
res_pop = evaluate(pop_model.score, data["val_candidates"], cold_items=data["cold_items"],
                   user_activity=data["user_activity"], verbose=False)

def flatten(r):
    """İç içe sonuç sözlüğünü satırlara açar (genel, düşük, orta, yüksek ...)."""
    rows = {}
    if "hit_rate" in r:
        rows["genel"] = {k: v for k, v in r.items() if not isinstance(v, dict)}
    def walk(d, prefix=""):
        for key, val in d.items():
            if isinstance(val, dict):
                if "hit_rate" in val:
                    rows[prefix + str(key)] = val
                else:
                    walk(val, prefix + str(key) + " / ")
    walk(r)
    return pd.DataFrame(rows).T

cols = ["hit_rate", "ndcg", "coverage_pct", "n_users"]
comparison = pd.concat({"Popülerlik": flatten(res_pop)[cols],
                        "Item-kNN": flatten(res)[cols]}, axis=1)
comparison.round(4)

[PopularityModel] fit tamamlandi: 25,610 urun, max_pop=1901, pozitif_urun=25,367


Popülerlik                               Item-kNN          \
                    hit_rate    ndcg coverage_pct  n_users hit_rate    ndcg   
overall               0.5488  0.3369      19.8438  75799.0   0.6220  0.4407   
cold_user             0.0000  0.0000       3.8735    114.0   0.0000  0.0000   
activity / low        0.5582  0.3441      18.1843  24415.0   0.6195  0.4314   
activity / medium     0.5541  0.3415      18.4381  30042.0   0.6188  0.4396   
activity / high       0.5305  0.3221      18.1374  21342.0   0.6294  0.4529   

                                         
                  coverage_pct  n_users  
overall                93.8891  75799.0  
cold_user               4.1351    114.0  
activity / low         74.7716  24415.0  
activity / medium      84.4280  30042.0  
activity / high        85.9898  21342.0

In [4]:
import itertools
from pathlib import Path
Path("results").mkdir(exist_ok=True)

def run_val(model):
    r = evaluate(model.score, data["val_candidates"], cold_items=data["cold_items"],
                 user_activity=data["user_activity"], verbose=False)
    f = flatten(r)
    return {"hit_rate": f.loc["overall", "hit_rate"], "ndcg": f.loc["overall", "ndcg"],
            "hr_low": f.loc["activity / low", "hit_rate"],
            "hr_medium": f.loc["activity / medium", "hit_rate"],
            "hr_high": f.loc["activity / high", "hit_rate"]}

rows = []
for k, shrink in itertools.product([500, 1000, 2000], [10, 30, 100]):
    m = ItemKNNModel(k=k, shrink=shrink, pop_weight=1e-3).fit(R)
    t = time.time()
    rows.append({"k": k, "shrink": shrink, "pop_weight": 1e-3, **run_val(m),
                 "fit_s": round(m.fit_seconds, 1), "eval_s": round(time.time() - t, 1)})
    print(rows[-1])

grid1 = pd.DataFrame(rows).sort_values("ndcg", ascending=False)
grid1.to_csv("results/itemknn_val_grid_stage1.csv", index=False)
grid1.round(4)

{'k': 500, 'shrink': 10, 'pop_weight': 0.001, 'hit_rate': np.float64(0.625641499228222), 'ndcg': np.float64(0.4411476324557427), 'hr_low': np.float64(0.6201515461806266), 'hr_medium': np.float64(0.6267891618400906), 'hr_high': np.float64(0.6303064380095587), 'fit_s': 8.9, 'eval_s': 29.2}
{'k': 500, 'shrink': 30, 'pop_weight': 0.001, 'hit_rate': np.float64(0.6266573437644296), 'ndcg': np.float64(0.44356940174307136), 'hr_low': np.float64(0.6204792135981978), 'hr_medium': np.float64(0.6277877637973504), 'hr_high': np.float64(0.6321338206353668), 'fit_s': 10.0, 'eval_s': 26.5}
{'k': 500, 'shrink': 100, 'pop_weight': 0.001, 'hit_rate': np.float64(0.627488489294054), 'ndcg': np.float64(0.4467814013536178), 'hr_low': np.float64(0.620602088879787), 'hr_medium': np.float64(0.6285533586312496), 'hr_high': np.float64(0.6338674913316465), 'fit_s': 8.0, 'eval_s': 29.3}
{'k': 1000, 'shrink': 10, 'pop_weight': 0.001, 'hit_rate': np.float64(0.6220134830274806), 'ndcg': np.float64(0.4407005048848438),

,k,shrink,pop_weight,hit_rate,ndcg,hr_low,hr_medium,hr_high,fit_s,eval_s
8,2000,100,0.001,0.6250,0.4476,0.6179,0.6214,0.6383,19.8,67.5
5,1000,100,0.001,0.6257,0.4474,0.6209,0.6228,0.6353,16.8,30.2
2,500,100,0.001,0.6275,0.4468,0.6206,0.6286,0.6339,8.0,29.3
1,500,30,0.001,0.6267,0.4436,0.6205,0.6278,0.6321,10.0,26.5
4,1000,30,0.001,0.6238,0.4435,0.6203,0.6209,0.6318,19.0,65.3
7,2000,30,0.001,0.6211,0.4431,0.6143,0.6175,0.6339,18.2,66.8
0,500,10,0.001,0.6256,0.4411,0.6202,0.6268,0.6303,8.9,29.2
3,1000,10,0.001,0.6220,0.4407,0.6195,0.6188,0.6294,9.5,30.3
6,2000,10,0.001,0.6192,0.4403,0.6129,0.6154,0.6318,8.5,28.2


In [5]:
best = grid1.iloc[0]
print("Aşama 1 en iyisi:", int(best["k"]), float(best["shrink"]))

m = ItemKNNModel(k=int(best["k"]), shrink=float(best["shrink"])).fit(R)
rows = []
for pw in [0, 1e-4, 1e-3, 1e-2, 3e-2]:
    m.pop_weight = pw
    rows.append({"k": int(best["k"]), "shrink": float(best["shrink"]), "pop_weight": pw, **run_val(m)})
    print(rows[-1])

grid2 = pd.DataFrame(rows)
grid2.to_csv("results/itemknn_val_grid_stage2.csv", index=False)
grid2.round(4)

Aşama 1 en iyisi: 2000 100.0
{'k': 2000, 'shrink': 100.0, 'pop_weight': 0, 'hit_rate': np.float64(0.5713663768651301), 'ndcg': np.float64(0.42102981533237094), 'hr_low': np.float64(0.5232439074339545), 'hr_medium': np.float64(0.573497104054324), 'hr_high': np.float64(0.6234186111892044)}
{'k': 2000, 'shrink': 100.0, 'pop_weight': 0.0001, 'hit_rate': np.float64(0.6198894444517737), 'ndcg': np.float64(0.44164099331587714), 'hr_low': np.float64(0.6131066967028466), 'hr_medium': np.float64(0.6162039810931362), 'hr_high': np.float64(0.6328366601068316)}
{'k': 2000, 'shrink': 100.0, 'pop_weight': 0.001, 'hit_rate': np.float64(0.6250214382775499), 'ndcg': np.float64(0.44760536725348715), 'hr_low': np.float64(0.6178988326848249), 'hr_medium': np.float64(0.6213967112708875), 'hr_high': np.float64(0.638271952019492)}
{'k': 2000, 'shrink': 100.0, 'pop_weight': 0.01, 'hit_rate': np.float64(0.6400216361693426), 'ndcg': np.float64(0.45202701226824404), 'hr_low': np.float64(0.6299406102805652), 'hr_m

,k,shrink,pop_weight,hit_rate,ndcg,hr_low,hr_medium,hr_high
0,2000,100.0,0.0000,0.5714,0.4210,0.5232,0.5735,0.6234
1,2000,100.0,0.0001,0.6199,0.4416,0.6131,0.6162,0.6328
2,2000,100.0,0.0010,0.6250,0.4476,0.6179,0.6214,0.6383
3,2000,100.0,0.0100,0.6400,0.4520,0.6299,0.6389,0.6532
4,2000,100.0,0.0300,0.6341,0.4309,0.6231,0.6343,0.6464


In [6]:
# En iyi validation ayarıyla modeli kaydet ve Popülerlik ile karşılaştır
best_row = grid2.sort_values("ndcg", ascending=False).iloc[0]
best_k = int(best_row["k"])
best_shrink = float(best_row["shrink"])
best_pw = float(best_row["pop_weight"])
print(f"Seçilen en iyi Item-kNN: k={best_k}, shrink={best_shrink}, pop_weight={best_pw}")

best_model = ItemKNNModel(k=best_k, shrink=best_shrink, pop_weight=best_pw).fit(R)
best_model.save("saved_models/itemknn")

res_best = evaluate(best_model.score, data["val_candidates"], cold_items=data["cold_items"],
                    user_activity=data["user_activity"], verbose=False)

final_val_comp = pd.concat({
    "Popülerlik": flatten(res_pop)[cols],
    "Item-kNN (En İyi Val)": flatten(res_best)[cols],
}, axis=1)
final_val_comp.round(4)

Seçilen en iyi Item-kNN: k=2000, shrink=100.0, pop_weight=0.01


                  Popülerlik                         Item-kNN (En İyi Val)  \
                    hit_rate    ndcg coverage_pct  n_users              hit_rate   
overall               0.5488  0.3369      19.8438  75799.0                0.6400   
cold_user             0.0000  0.0000       3.8735    114.0                0.0000   
activity / low        0.5582  0.3441      18.1843  24415.0                0.6299   
activity / medium     0.5541  0.3415      18.4381  30042.0                0.6389   
activity / high       0.5305  0.3221      18.1374  21342.0                0.6532   

                                                  
                     ndcg coverage_pct  n_users  
overall            0.4520      91.0426  75799.0  
cold_user          0.0000       4.0570    114.0  
activity / low     0.4417      66.4701  24415.0  
activity / medium  0.4527      76.9231  30042.0  
activity / high    0.4628      76.8840  21342.0  